# Save category comment-only preprocessors

Run this after the final category comment-only model is trained on the same dataset. It saves only preprocessing objects, not the model.

In [ ]:
from pathlib import Path
import re
import joblib
import nltk
import pandas as pd
import pymorphy3
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')

In [ ]:
DATA_CANDIDATES = [
    Path('nerd_analytics_v25.xlsx'),
    Path('..') / 'nerd_analytics_v25.xlsx',
    Path('..') / '..' / 'nerd_analytics_v25.xlsx',
]

DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), DATA_CANDIDATES[0])
SHEET_NAME = 'reviews'
OUTPUT_PATH = Path('reviews_categories_comment_only_preprocessors.pkl')

TEXT_COLUMN = 'comment'
TARGET_COLUMN = 'final_category'
BAD_STOPWORDS = ['не', 'нет', 'ни', 'без']

print(f'Using data path: {DATA_PATH}')

In [ ]:
morph = pymorphy3.MorphAnalyzer()
russian_stopwords = [
    word for word in stopwords.words('russian')
    if word not in BAD_STOPWORDS
]

CLEAN_TEXT_SOURCE = r'''
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\\S+", " ", text)
    text = re.sub(r"www\\S+", " ", text)
    text = re.sub(r"@\\w+", " ", text)
    text = re.sub(r"[^а-яa-zё\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    words = text.split()
    lemmas = []

    for word in words:
        if word not in russian_stopwords:
            lemma = morph.parse(word)[0].normal_form
            lemmas.append(lemma)

    return " ".join(lemmas)
'''

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^а-яa-zё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    lemmas = []

    for word in words:
        if word not in russian_stopwords:
            lemma = morph.parse(word)[0].normal_form
            lemmas.append(lemma)

    return " ".join(lemmas)

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df = df[['id', 'comment', 'sentiment', 'rating', 'keywords_positive', 'keywords_neutral', 'keywords_negative', 'product', 'final_category']]

cat_df = df[[TEXT_COLUMN, TARGET_COLUMN]].copy()
cat_df['clean_comment'] = cat_df[TEXT_COLUMN].apply(clean_text)

word_vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2, max_df=0.95)
char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=20000)

X_word = word_vectorizer.fit_transform(cat_df['clean_comment'])
X_char = char_vectorizer.fit_transform(cat_df['clean_comment'])

bundle = {
    'task': 'category_comment_only',
    'model_file': 'reviews_categories_comment_only.pkl',
    'text_column': TEXT_COLUMN,
    'target_column': TARGET_COLUMN,
    'word_vectorizer': word_vectorizer,
    'char_vectorizer': char_vectorizer,
    'preprocessing': {
        'bad_stopwords': BAD_STOPWORDS,
        'russian_stopwords': russian_stopwords,
        'clean_text_source': CLEAN_TEXT_SOURCE,
        'morph_analyzer': 'pymorphy3.MorphAnalyzer()',
    },
    'metadata': {
        'data_path': str(DATA_PATH),
        'sheet_name': SHEET_NAME,
        'n_rows': len(cat_df),
        'word_features': X_word.shape[1],
        'char_features': X_char.shape[1],
        'feature_order': ['word_tfidf', 'char_tfidf'],
    },
}

joblib.dump(bundle, OUTPUT_PATH)
print(f'Saved preprocessors to: {OUTPUT_PATH}')
print(bundle['metadata'])